In [ ]:
import numpy as np
import torch
import smplx
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from chumpy.utils import row, col
import trimesh

def graph_laplacian(graph):
    row, col, data = [], [], []
    for v in graph:
        n = len(graph[v])
        row += [v] * n
        col += [u for u in graph[v]]
        data += [1.0 / n] * n
    return coo_matrix((data, (row, col)), shape=[len(graph)] * 2)

def laplacian_matrix(faces):
    G = {}
    for face in faces:
        for i, v in enumerate(face):
            nv = face[(i + 1) % len(face)]
            if v not in G:
                G[v] = {}
            if nv not in G:
                G[nv] = {}
            G[v][nv] = 1
            G[nv][v] = 1
    return graph_laplacian(G)

def update_clothskinweight(v_template,faces,smpl_v, smooth_num=200, bw =  None):
    tree = cKDTree(smpl_v.cpu().numpy())
    # smplbw = np.load(global_var.BW_PATH, allow_pickle=True)
    smplbw = bw
    _, idx = tree.query(v_template, workers=-1)
    # NOTE: 骨骼索引
    # input_joints = [0, 1, 2, 3, 4, 5, 6, 9, 12, 13, 14, 16, 17, 18, 19]
    input_joints = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
    blend_weights = smplbw[idx][:, input_joints]
    blend_weights /= blend_weights.sum(1, keepdims=True)
    
    clothfaces = faces
    lap = laplacian_matrix(clothfaces)
    
    # for _ in range(200):
    for _ in range(smooth_num):
        blend_weights = lap @ blend_weights
    
    return blend_weights.astype(np.float32)

# 选择男性或女性
smpl_model = smplx.SMPL("../data/SMPL/SMPL_FEMALE.pkl")
bw = smpl_model.lbs_weights.numpy()

# 需要计算权重的服装
garment = trimesh.load("../data/magdalena/magdalena2000-allviews/templatedeformT/deformmodel.obj")

smploutput = smpl_model(
    # betas = torch.tensor([-0.4673,  1.1931,  0.5731, -0.4866,  0.5409,  0.4370,  2.5331,  0.8447, -2.3165, -0.5910],dtype=torch.float32).reshape(-1, 10),
    return_verts = True
)

smpl_vertices = smpl_model.v_template

# smooth次数
smooth = 50
clothbw = update_clothskinweight(garment.vertices, garment.faces, smpl_vertices, smooth, bw)
np.save(f"../data/magdalena/magdalena2000-allviews/bw_weight.npy", clothbw)